In [19]:
import pandas as pd
import numpy as np
df=pd.read_csv('all_kindle_review.csv')

In [20]:
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


  About Dataset
  Context This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

  Content 5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset. Columns

  asin - ID of the product, like B000FA64PK


  helpful - helpfulness rating of the review - example: 2/3.


  overall - rating of the product.
  reviewText - text of the review (heading).


  reviewTime - time of the review (raw).


  reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN


  reviewerName - name of the reviewer.
  summary - summary of the review (description).

  
  unixReviewTime - unix timestamp.
  Acknowledgements This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

In [21]:
### prepocessing and data cleaning foe the features

In [22]:
data=df[['reviewText','rating']]

In [23]:
data.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [24]:
# checking the imbalance dataset
# data['rating'].unique()
data['rating'].value_counts()

,count
rating,
5,3000
4,3000
3,2000
2,2000
1,2000


In [26]:
###preprocessing and cleaning
data['rating']=data['rating'].apply(lambda x:0 if x<3 else 1)

/tmp/ipykernel_1260/3167557681.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['rating']=data['rating'].apply(lambda x:0 if x<3 else 1)


In [27]:
df=data

In [28]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",1
1,Great short read. I didn't want to put it dow...,1
2,I'll start by saying this is the first of four...,1
3,Aggie is Angela Lansbury who carries pocketboo...,1
4,I did not expect this type of book to be in li...,1


In [30]:
df['rating'].value_counts()

,count
rating,
1,8000
0,4000


In [34]:
### text data preprocessing
# 1) lower all the charecter in the string
df['reviewText']=df['reviewText'].str.lower()

In [35]:
import re

In [38]:
# 2) removing the url
df['reviewText']=df['reviewText'].apply(lambda x: re.sub('https?://\S+|www\.\S+','',x))

<>:2: SyntaxWarning: invalid escape sequence '\S'
<>:2: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_1260/4042184302.py:2: SyntaxWarning: invalid escape sequence '\S'
  df['reviewText']=df['reviewText'].apply(lambda x: re.sub('https?://\S+|www\.\S+','',x))


In [39]:
## removing the special charecters
df['reviewText']=df['reviewText'].apply(lambda x: re.sub('A-Za-z0-9+','',x))

In [40]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [43]:
###removing the stopwords from the sentences
df['reviewText']=df['reviewText'].apply(lambda x:" ".join(y for y in  x.split() if y not in stopwords.words('english')))

In [44]:
df.head()

,reviewText,rating
0,"jace rankin may short, nothing mess with, man ...",1
1,great short read. want put read one sitting. s...,1
2,start saying first four books expecting &#34;c...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [52]:
from nltk.stem import WordNetLemmatizer
wl=WordNetLemmatizer()
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [54]:
### appying the lemmatizer
df['reviewText']=df['reviewText'].apply(lambda x : " ".join(wl.lemmatize(y,pos='v') for y in x.split() ))

In [55]:
### Train TEst Split
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(df['reviewText'],df['rating'],test_size=0.20)

In [60]:
x_train.shape,y_test.shape

((9600,), (2400,))

In [68]:
## now we have havve to convert the words into vectors such that model should be able to understand it.
## we have BOG,TFIDF,Word2Vec
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
x_train_bow=bow.fit_transform(x_train).toarray()
x_test_bow=bow.transform(x_test).toarray()

In [69]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
x_train_tfidf=tfidf.fit_transform(x_train).toarray()
x_test_tfidf=tfidf.transform(x_test).toarray()

In [75]:
### since it is a classification algorithm we use classification algorithms
from sklearn.naive_bayes import GaussianNB
gnb=GaussianNB()
nb_model_bow=gnb.fit(x_train_bow,y_train)
nb_model_tfidf=gnb.fit(x_train_tfidf,y_train)

In [76]:
from sklearn.metrics import classification_report,accuracy_score
y_pred_bow=gnb.predict(x_test_bow)
y_pred_tfidf=gnb.predict(x_test_tfidf)

In [77]:
def evaluate(model_name,y_test,y_pred):
  print(f'Classification report of model {model_name} is {classification_report(y_test,y_pred)}')
  print(f'Accuracy score of model {model_name} is {accuracy_score(y_test,y_pred)}')



In [81]:
evaluate('BOW',y_test,y_pred_bow)


Classification report of model BOW is               precision    recall  f1-score   support

           0       0.45      0.59      0.51       789
           1       0.76      0.65      0.70      1611

    accuracy                           0.63      2400
   macro avg       0.60      0.62      0.60      2400
weighted avg       0.66      0.63      0.64      2400

Accuracy score of model BOW is 0.62625


In [80]:
evaluate('Tfidf',y_test,y_pred_tfidf)

Classification report of model Tfidf is               precision    recall  f1-score   support

           0       0.41      0.63      0.49       789
           1       0.75      0.55      0.63      1611

    accuracy                           0.57      2400
   macro avg       0.58      0.59      0.56      2400
weighted avg       0.64      0.57      0.59      2400

Accuracy score of model Tfidf is 0.5745833333333333


since the dataset is huge we should be using the word2vec for this and we will se the accuracy then

In [84]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 14.1 MB/s eta 0:00:00


In [89]:
from gensim.models import Word2Vec
import numpy as np

# 1) Tokenize your text (Word2Vec needs a list of tokenized sentences, not raw strings)
x_train_tokens = [text.split() for text in x_train]
x_test_tokens = [text.split() for text in x_test]

# 2) Train Word2Vec ONLY on training data (avoid leakage, same principle as before)
w2v_model = Word2Vec(sentences=x_train_tokens, vector_size=100, window=5, min_count=1, workers=4)

# 3) Function to average word vectors for each document
def get_avg_word2vec(tokens, model, vector_size):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

# 4) Apply to train and test sets (using the SAME trained model for both)
x_train_word2vec = np.array([get_avg_word2vec(tokens, w2v_model, 100) for tokens in x_train_tokens])
x_test_word2vec = np.array([get_avg_word2vec(tokens, w2v_model, 100) for tokens in x_test_tokens])

In [91]:
nb_model_w2vec=gnb.fit(x_train_word2vec,y_train)
y_pred_w2v=nb_model_w2vec.predict(x_test_word2vec)
evaluate('Word2Vec',y_test,y_pred_w2v)

Classification report of model Word2Vec is               precision    recall  f1-score   support

           0       0.43      0.74      0.54       789
           1       0.80      0.52      0.63      1611

    accuracy                           0.59      2400
   macro avg       0.61      0.63      0.59      2400
weighted avg       0.68      0.59      0.60      2400

Accuracy score of model Word2Vec is 0.59
